# IZV část 3 projektu- Test hypotézy

#### Autor: Karolína Virágová, xvirag00

Úkolem je ověřit následující dvě hypotézy s 95% jistotou.

In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
from scipy.stats import shapiro, ttest_ind, mannwhitneyu

In [26]:
accidents = pd.read_pickle('accidents.pkl.gz', compression='gzip')
vehicles = pd.read_pickle('vehicles.pkl.gz', compression='gzip')

In [27]:
 # 95% interval spolehlivosti
alpha = 0.05 

## Hypotéza 1
Na silnicích první třídy se byly nehody s následky na zdraví se stejnou pravděpodobností
jako na dálnicích.



Nejprve si vyfiltrujema data jen pro dálnice a silnice 1. třídy

In [28]:
# Filtrování dat pro dálnice a silnice 1. třídy
highways = accidents[(accidents['p36'] == 0)| (accidents['p36'] == 1)]

Vytvoříme kontingenčbí tabulku

In [29]:
contingency_table = pd.crosstab(
    highways['p36'] == 1,
    highways['p9'] == 1,
)

print("Kontingenční tabulka:")
print(contingency_table)

Kontingenční tabulka:
p9     False  True 
p36                
False   6674   1247
True   14773   7059


Provedeme  χ² test

In [30]:
# Provedení χ² testu
chi2_stat, p_value, dof, expected = stats.chi2_contingency(
    contingency_table
    )

# Výsledky testu
print("\nVýsledky χ² testu:")
print(f"Testová statistika (χ²): {chi2_stat}")
print(f"P-hodnota: {p_value}")
print(f"Stupně volnosti (dof): {dof}")
print("Očekávané hodnoty:")
print(expected)


Výsledky χ² testu:
Testová statistika (χ²): 794.1532980454858
P-hodnota: 1.0075002730159729e-174
Stupně volnosti (dof): 1
Očekávané hodnoty:
[[ 5709.73303532  2211.26696468]
 [15737.26696468  6094.73303532]]


### Výsledek hypotézy:  
Po spuštění kódu se vypíše zda zamítáme nulovou hypotézu a k tomu výsledek nebo se vypíše, že se nepodařilo zamítnout nulovou hypotézu a výsledek. Co znamená výsledek je popsané níže.

In [31]:
# Interpretace výsledků
print("\nVýsledek testu:")
if p_value < alpha:
    print("Zamítáme nulovou hypotézu.")
    if expected[0][0] > contingency_table.iloc[0, 0]:
        print("Výsledek A.")
    else:
        print("Výsledek B.")
else:
    print("Nepodařilo se zamítnout nulovou hypotézu.")
    print("Výsledek C.")



Výsledek testu:
Zamítáme nulovou hypotézu.
Výsledek B.


#### Zamítáme nulovou hypotézu
Výsledek A. Nehody na silnicích 1. třídy vedly častěji k následkům na zdraví než na dálnicích.  
Výsledek B. Nehody na dálnicích vedly častěji k následkům na zdraví než na silnicích 1. třídy

#### Nepodařilo se zamítnout nulovou hypotézu.
Výsledek C. Nehody na silnicích 1. třídy a na dálnicích mají podobnou pravděpodobnost následků na zdraví.

## Hypotéza 2
Škoda při nehodách trolejbusů je nižší, než při nehodách autobusů a tato odchylka je
statisticky významná.

In [32]:
# Sloučení datových sad
merged = pd.merge(vehicles, accidents, on="p1", how="inner")

# Filtrování na nehody trolejbusů a autobusů
trolleybus = merged[merged["p44"] == 11] 
bus = merged[merged["p44"] == 8] 
# Extrahování škod
trolleybus_damage = trolleybus["p53"].dropna()
bus_damage = bus["p53"].dropna()

Zjistíme zda jsou distibuce škody na autobuse a trolejbuse normálně rozděleny.
Provedeme Shapiro-Wilk test.

In [33]:
# Testování normality
stat_trolley, p_trolley = shapiro(trolleybus_damage)
stat_bus, p_bus = shapiro(bus_damage)

Jestliže nejsou normálně rozloženy, Použijeme  Mann-Whitneyho U test.
Jinak použijeme Studentův t-test.

In [34]:
if p_trolley < alpha or p_bus < alpha:
    stat, p_value = mannwhitneyu(
        trolleybus_damage, bus_damage, alternative="less"
        )
else:
    stat, p_value = ttest_ind(
        trolleybus_damage, bus_damage, alternative="less"
        )

### Výsledek hypotézy:
Po spuštění kódu se vypíše výsledek. Co znamená výsledek je popsané níže.

In [35]:
# Výstup testu
print("\nVýsledek testu:")
# Interpretace
if p_value < alpha:
    print("Výsledek A.")
else:
    print("Výsledek B")


Výsledek testu:
Výsledek A.


#### Výsledek A
Hypotézu zamítáme: Škoda při nehodách trolejbusů je nižší než při nehodách autobusů.

#### Výsledek B
Hypotézu nezamítáme: Neexistuje dostatek důkazů, že škoda při nehodách trolejbusů je nižší.